In [1]:
%matplotlib tkagg

In [2]:
import os as os
os.getcwd()

'/Users/John/Dev/PythonScripts/DataGopher2/srcnew'

In [3]:
run datagopher.py

/Users/John/Dev/PythonScripts/DataGopher2/srcnew/datagopher.py:196: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath, skiprows = 1, header=None) # header=0 is default
/Users/John/miniconda3/envs/jptr/lib/python3.14/site-packages/seaborn/axisgrid.py:123: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  self._figure.tight_layout(*args, **kwargs)


In [ ]:
list(globals())

In [ ]:
global_names = list(globals())
print(global_names)
gdata in global_names
gdata = globals()['gdata']
gdata.data.head(10)

In [2]:
run ~/Dev/PythonScripts/getData.py

2025-11-29 16:13:19.542 python[82558:41570319] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'


In [3]:
globals()['gdata'].data.head(10)

,Unnamed: 0,PRICE,ZIP,SQFT,PRICE_PER_SQFT,BEDS,BATHS,CITY,LATITUDE,LONGITUDE,YEAR_BUILT
0,0,426000,48108,1908.0,223.0,3.0,3.5,Ann Arbor,42.237403,-83.688046,1995.0
1,1,225000,48108,1080.0,208.0,3.0,1.5,Ann Arbor,42.234612,-83.716912,1971.0
2,2,470000,48108,3000.0,157.0,30.0,45.0,Ann Arbor,42.240761,-83.704096,2015.0
3,3,355000,48108,1637.0,217.0,3.0,2.5,Ann Arbor,42.250912,-83.668212,1995.0
4,4,684860,48108,3500.0,196.0,4.0,3.5,Pittsfield Twp,42.203995,-83.700493,2023.0
5,5,655000,48108,3276.0,200.0,5.0,2.5,Ann Arbor,42.198648,-83.758332,1988.0
6,6,315000,48108,864.0,365.0,3.0,2.0,Ann Arbor,42.234116,-83.713424,1971.0
7,7,475000,48108,3726.0,127.0,4.0,3.5,Ann Arbor,42.229417,-83.770141,1996.0
8,8,460000,48108,2018.0,228.0,4.0,2.5,Ann Arbor,42.211470,-83.696103,2004.0
9,9,109000,48108,864.0,126.0,3.0,1.0,Ann Arbor,42.233992,-83.710067,1971.0


In [10]:
import pandas as pd
from pandas.api.types import is_numeric_dtype
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
matplotlib.use('TkAgg')
import numpy as np
import statsmodels.api  as sm
import statsmodels.formula.api as smf 
from statsmodels.graphics.regressionplots import plot_partregress_grid, plot_leverage_resid2, influence_plot, plot_fit
from patsy import dmatrices, dmatrix, NAAction
import seaborn as sb
from sklearn.metrics import roc_curve, auc


basecolors0 = ['blue', 'red',  'green', 'yellow', 'magenta', 'cyan', 'violet',
               'orange',  'goldenrod', 'grey', 'gold', 'silver', 'orangered', 'darkolivegreen',
               'olive', 'khaki', 'thistle', 'lightsteelblue', 'slateblue', 'black', 'darkviolet',
               'brown', 'indigo', 'hotpink', 'lavender']


def getcolor(cvar, col_data):
    dfc = pd.DataFrame(col_data)
    choicesCo = list(dfc[dfc.columns[0]].unique())
    choicesCo.sort()
    if (len(choicesCo) < len(basecolors0)):
        colorD = {item : basecolors0[choicesCo.index(item)]  for item in choicesCo}
        colorlist = [colorD[item] for item in col_data]
        lpatches = [mpatches.Patch(color = colorD[item],label = cvar + ', ' + str(item)) for item in colorD.keys()]
    else:
        cmap = plt.cm.plasma
        colorD = {item : cmap(choicesCo.index(item)/len(choicesCo)) for item in choicesCo}
        colorlist = [colorD[item] for item in col_data]
        lpatches = [mpatches.Patch(color = colorD[item],label = cvar +  ', ' + str(item)) for item in choicesCo]
    return colorD, colorlist, lpatches

#############################
### Model Plotting Functions
#############################


def doroc(MODEL = None):
    if (MODEL is None): return
    prediction_res = MODEL.get_prediction(transform = True)
    res_frame= prediction_res.summary_frame(alpha = 0.05)

    fpr, tpr, thresholds = roc_curve(MODEL.model.endog, res_frame['mean']) 
    roc_auc = auc(fpr, tpr)
    fig = plt.figure(figsize = (8,8))
    ax = fig.add_subplot()
    ax.plot(fpr, tpr)
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f"ROC: {MODEL.model.formula}, AUC={round(roc_auc,5)}")
    fig.show()  #show the plots simultaneously
    #plt.show() #show plots one at a time
    return

def dopredictbin(res = None):
    if res is None: return
    xvpredictlist = list(res.model.exog_names)
    cvpredict = res.model.endog
    tstr = res.model.formula
    dsize = 8.0
    prediction_res = res.get_prediction(transform = True)
    res_frame= prediction_res.summary_frame(alpha = 0.05)
    ylabstr = "Est. Mean Response"
 
    for idx,xvpredict in enumerate(xvpredictlist):
        if idx == 0: continue
        fig, ax = plt.subplots() 
        sb.scatterplot(ax= ax, x = res.model.exog[:,idx], y = res_frame['mean'], hue = res.model.endog, palette = 'bright',s = dsize)
        plt.axhline(y=0, color='black', linestyle='-')
        plt.axhline(y=0.2, color = 'black', linestyle = 'dashed',linewidth = 0.5)
        plt.axhline(y=0.4, color = 'black', linestyle = 'dashed',linewidth = 0.5)
        plt.axhline(y=0.5, color = 'black', linestyle = 'solid',linewidth = 0.5)
        plt.axhline(y=0.6, color = 'black', linestyle = 'dashed',linewidth = 0.5)
        plt.axhline(y=0.8, color = 'black', linestyle = 'dashed',linewidth = 0.5)
        plt.axhline(y=1.0, color = 'black', linestyle = '-',linewidth = 0.5)
        plt.ylim((0, 1))
        plt.ylabel(ylabstr)
        plt.xlabel(xvpredict)
        plt.title(tstr)
        fig.show()  #show the plots simultaneously
        #plt.show() #show plots one at a time
    return

def dopredict(res = None):
    if res is None: return
    xvpredictlist = list(res.model.exog_names)
    tstr = res.model.formula
    depvar = res.model.endog
    dsize = 8.0
    prediction_res = res.get_prediction(transform = True)
    res_frame= prediction_res.summary_frame(alpha = 0.05)
    ylabstr = "Est. Mean Response"
    for idx,xvpredict in enumerate(xvpredictlist):
        if idx == 0: continue
        fig, ax = plt.subplots() 
        sb.scatterplot(ax = ax, x = res.model.exog[:,idx], y = depvar, color = 'blue',label = 'Observed', s= dsize)
        sb.scatterplot(ax = ax, x = res.model.exog[:,idx], y = res_frame['mean'], color = 'red',label = 'Predicted', s = dsize)
        #plt.ylim((0, 1))
        plt.ylabel(ylabstr)
        plt.xlabel(xvpredict)
        plt.title(tstr)
        fig.show()  #show the plots simultaneously
        #plt.show() #show plots one at a time
    return

def doresidual(res = None, mtype = 'OLS'):
    if res == None: return
    tstr = res.model.formula
    xvresidlist = list(res.model.exog_names)
    dsize = 8 #dot size
    if mtype != 'OLS': #if it's not OLS it's GLM
        vres = res.resid_deviance
        ylabstr = 'Deviance Residual'
    else:
        vres = res.resid
        ylabstr = 'Residual'
    residlim = max(np.abs(vres))
    for idx, xvresid in enumerate(xvresidlist):
        fig, ax = plt.subplots()
        if idx == 0 :
            prediction_res = res.get_prediction(transform = True)
            res_frame= prediction_res.summary_frame(alpha = 0.05)            
            sb.scatterplot(ax=ax, x = res_frame['mean'], y = vres, color = 'blue', s = dsize)
            plt.xlabel('Predicted ' + res.model.endog_names)
        else:
            sb.scatterplot(ax=ax, x = res.model.exog[:,idx], y = vres, color = 'blue', s = dsize) 
            plt.xlabel(xvresid)
        plt.axhline(y=0, color='black', linestyle='-')
        plt.ylim((-residlim, residlim))
        plt.ylabel(ylabstr)

        plt.title(tstr)
        fig.show()  #show the plots simultaneously
        #plt.show() #show plots one at a time
    return

#############################
#############################


In [11]:
df = pd.read_csv('/Users/John/Dev/PythonScripts/DataFiles/AnnArborRealEstate_noBedoutliers_noLotoutliers-b.csv',engine='python')
 
 
df = pd.read_csv('/Users/John/Dev/PythonScripts/DataFiles/AnnArborRealEstate_noBedoutliers_noLotoutliers-b.csv',engine='python')
nobs0 = len(df)
df.dropna(subset=['ZIP', 'YEAR_BUILT', 'SQFT'], inplace = True)
nobs = len(df)
df.dropna(subset = ['ZIP', 'YEAR_BUILT', 'SQFT'], inplace = True)
sfig = plt.figure(figsize = (8,8)) 
ax = sfig.subplots()
sb.scatterplot(df, ax = ax, **{'x': 'YEAR_BUILT', 'y': 'SQFT', 'hue': 'ZIP', 'palette': 'bright', 'style': None, 'color': None, 's': 5, 'marker': 'o'})
smoothed = sm.nonparametric.lowess(exog=np.array(df['YEAR_BUILT']), endog=np.array(df['SQFT']), frac=0.3)
dfsmooth = pd.DataFrame({"xs": smoothed[:, 0], "ys": smoothed[:, 1]})
sb.lineplot(data = dfsmooth, ax = ax, **{'x': 'xs', 'y': 'ys', 'color': 'darkviolet', 'linewidth': 1.0, 'label': 'SQFT', 'markers': False, 'estimator': None, 'errorbar': None, 'linestyle': 'dashdot'})
ax.set_xlabel('YEAR_BUILT')
ax.set_ylabel('SQFT')
ax.legend(bbox_to_anchor=(0.98, 1), loc='upper left', borderaxespad=2)
ax.set_xlim(1887.0, 2024.0)
ax.set_ylim(627.0, 6965.0)
sfig.suptitle(t='(872 rows out of 873)', y=0.95)
sfig.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)
sfig.show()
#plt.show()


In [12]:
len(df)

872